<a href="https://colab.research.google.com/github/anthony-yt/deepwater-image-segmentation-classical-quantum/blob/feat%2Flabelling-evaluation-tools/notebooks/6_evaluation_harness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shared evaluation harness

In [2]:
import os
import subprocess
import sys
from pathlib import Path

import cv2
import numpy as np

REPO_URL = "https://github.com/anthony-yt/deepwater-image-segmentation-classical-quantum.git"
BRANCH = "feat/labelling-evaluation-tools"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Under_Water_Dataset")
    if not Path("src/evaluation").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, "/content/repo"], check=True)
        os.chdir("/content/repo")
else:
    DRIVE_ROOT = Path(os.environ.get("DW_DATA_ROOT", "Under_Water_Dataset")).resolve()

sys.path.insert(0, str(Path.cwd()))
from src.evaluation.segmentation_metrics import across_seeds, evaluate, frame_metrics

PROCESSED = DRIVE_ROOT / "Dataset_Processed"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Cases with a known answer

In [3]:
gt = np.zeros((100, 100), np.uint8)
gt[20:80, 20:80] = 255                                   # one 60 x 60 square: 36 % of the frame
shift = lambda m, d: np.roll(np.roll(m, d, axis=1), d, axis=0)   # diagonal shift of d px

perfect = frame_metrics(gt, gt)
empty = frame_metrics(np.zeros_like(gt), gt)
whole = frame_metrics(np.full_like(gt, 255), gt)
near = frame_metrics(shift(gt, 1), gt)                    # 1 px off: inside the 2 px tolerance
far = frame_metrics(shift(gt, 8), gt)                     # 8 px off: outside it

assert perfect["iou"] == perfect["dice"] == perfect["boundary_f1"] == 1.0
assert empty["iou"] == 0.0 and empty["recall"] == 0.0
assert abs(whole["iou"] - 0.36) < 1e-9 and whole["recall"] == 1.0
assert near["boundary_f1"] == 1.0 and far["boundary_f1"] < 0.1
assert frame_metrics(np.zeros_like(gt), np.zeros_like(gt))["iou"] == 1.0   # empty frame, empty prediction
print("known-answer cases: ok")

known-answer cases: ok


## 2. Real data: it must reproduce the ground-truth reconciliation numbers

In [4]:
def load_masks(directory):
    return {p.stem.removesuffix("_mask"): cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            for p in sorted(Path(directory).glob("*.png"))}

curated = load_masks(PROCESSED / "Test_Set_Curado")
sam_points = {k: v for k, v in load_masks(PROCESSED / "SAM_Masks_Frames_CLAHE").items() if k in curated}

per_frame, per_video = evaluate(sam_points, curated)
per_video.round(3)

,video,n_frames,iou,dice,precision,recall,boundary_f1,base_iou,iou_minus_base,iou_ci_low,iou_ci_high,diff_ci_low,diff_ci_high
0,Video1,20,0.664,0.787,0.742,0.874,0.040,0.718,-0.054,0.595,0.734,-0.111,0.001
1,Video6,2,0.067,0.125,0.067,0.999,0.054,0.064,0.003,0.050,0.085,0.002,0.004


In [5]:
row = per_video.iloc[0]
assert abs(row["iou"] - 0.664) < 0.002 and abs(row["base_iou"] - 0.718) < 0.002   # values from the ground-truth reconciliation notebook
print(f"v3 IoU {row['iou']:.3f}  |  everything-is-structure IoU {row['base_iou']:.3f}  |  "
      f"difference {row['iou_minus_base']:+.3f} [{row['diff_ci_low']:+.3f}, {row['diff_ci_high']:+.3f}]")

v3 IoU 0.664  |  everything-is-structure IoU 0.718  |  difference -0.054 [-0.111, +0.001]


## 3. Usage and seeds

In [6]:
# One line per model: predictions and ground truth as {frame_id: mask}.
per_frame, per_video = evaluate(sam_points, curated)

# With several seeds: one per_frame table per seed, then mean and sd over seeds for each video.
across_seeds({0: per_frame, 1: per_frame}).round(3)

,iou,dice,precision,recall,boundary_f1,base_iou,iou_sd,dice_sd,precision_sd,recall_sd,boundary_f1_sd,base_iou_sd,n_seeds
video,,,,,,,,,,,,,
Video1,0.664,0.787,0.742,0.874,0.040,0.718,0.0,0.0,0.0,0.0,0.0,0.0,2
Video6,0.067,0.125,0.067,0.999,0.054,0.064,0.0,0.0,0.0,0.0,0.0,0.0,2
